In [47]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score, adjusted_rand_score

# Пути
BASE = Path.cwd()
DATA_DIR = BASE / 'data'
ARTIFACTS = BASE / 'artifacts'
FIG_DIR = ARTIFACTS / 'figures'
LABELS_DIR = ARTIFACTS / 'labels'

ARTIFACTS.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
LABELS_DIR.mkdir(parents=True, exist_ok=True)

print('BASE', BASE)
print('DATA_DIR', DATA_DIR)
print('ARTIFACTS', ARTIFACTS)

BASE c:\Users\Maken\Desktop\AI-course\homeworks\HW07
DATA_DIR c:\Users\Maken\Desktop\AI-course\homeworks\HW07\data
ARTIFACTS c:\Users\Maken\Desktop\AI-course\homeworks\HW07\artifacts


In [48]:
def build_preprocessor(df):

    # числовые и категориальные признаки
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = [c for c in df.columns if c not in num_cols]

    num_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])

    cat_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

    preproc = ColumnTransformer([('num', num_pipe, num_cols), ('cat', cat_pipe, cat_cols)], remainder='drop')

    return preproc, num_cols, cat_cols


def compute_scores(X, labels):

    labels = np.asarray(labels)
    mask = labels != -1
    if mask.sum() < 2 or len(set(labels[mask])) < 2:
        return {'silhouette': None, 'db': None, 'ch': None, 'n_clusters': int(len(set(labels[mask]))), 'n_points': int(mask.sum()), 'noise_fraction': float((labels == -1).mean())}

    s = float(silhouette_score(X[mask], labels[mask]))
    db = float(davies_bouldin_score(X[mask], labels[mask]))
    ch = float(calinski_harabasz_score(X[mask], labels[mask]))

    return {'silhouette': s, 'db': db, 'ch': ch, 'n_clusters': int(len(set(labels[mask]))), 'n_points': int(mask.sum()), 'noise_fraction': float((labels == -1).mean())}


def plot_pca_scatter(X, labels, title, fname):

    pca = PCA(n_components=2)
    proj = pca.fit_transform(X)

    plt.figure(figsize=(6,5))
    sns.scatterplot(x=proj[:,0], y=proj[:,1], hue=labels, palette='tab10', s=10, legend=False)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(fname, dpi=150)
    plt.close()


def save_labels_df(df, labels, fname):

    # Сохраняем только идентификатор и метку кластера
    if 'sample_id' in df.columns:
        out = pd.DataFrame({'sample_id': df['sample_id'].values, 'cluster': labels})
    else:
        out = pd.DataFrame({'index': df.index.values, 'cluster': labels})
    out.to_csv(fname, index=False)


In [49]:
files = ['S07-hw-dataset-01.csv', 'S07-hw-dataset-02.csv', 'S07-hw-dataset-03.csv']

dfs = []
missing = []
for f in files:

    p = DATA_DIR / f
    if not p.exists():
        missing.append(f)
    else:
        dfs.append(pd.read_csv(p))

if missing:

    print('Отсутствуют файлы:', missing)
    while len(dfs) < 3:
        dfs.append(pd.DataFrame())

for i, df in enumerate(dfs, start=1):

    if not df.empty:
        df.columns = df.columns.str.strip()
        print(f'df_{i} shape:', df.shape)
    else:
        print(f'df_{i} is empty')

df_1, df_2, df_3 = dfs

df_1 shape: (12000, 9)
df_2 shape: (8000, 4)
df_3 shape: (15000, 5)


In [50]:
# Анализ каждого датасета: предобработка, KMeans, альтернатива, метрики, визуализация
summaries = {}

for idx, df in enumerate((df_1, df_2, df_3), start=1):
    ds_name = f'ds{idx}'

    if df.empty:
        summaries[ds_name] = {'note': 'empty'}
        continue

    # Быстрая разведка
    print('\n', '---', ds_name, '---')
    display(df.head())
    print(df.info())

    X = df.copy()
    if 'sample_id' in X.columns:
        X = X.drop(columns=['sample_id'])

    # Теперь строим препроцессор только по нужным признакам
    preproc, num_cols, cat_cols = build_preprocessor(X)
    X_proc = preproc.fit_transform(X)
    # ---------------------------------------

    # Поиск лучшего k по silhouette (k=2..6)
    best_k = None
    best_score = -1.0
    scores_by_k = {}
    
    # Защита: k не может быть больше количества строк - 1
    max_k = min(7, X_proc.shape[0])
    
    for k in range(2, max_k):
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(X_proc)
        try:
            s = silhouette_score(X_proc, labels)
        except Exception:
            s = -1
        scores_by_k[k] = float(s)
        if s > best_score:
            best_score = s
            best_k = k

    # Сохраним график silhouette vs k
    try:
        ks = list(scores_by_k.keys())
        vals = [scores_by_k[k] for k in ks]
        if ks:
            plt.figure(figsize=(5,4))
            plt.plot(ks, vals, marker='o')
            plt.xlabel('k')
            plt.ylabel('silhouette')
            plt.title(f'silhouette vs k {ds_name}')
            plt.tight_layout()
            plt.savefig(FIG_DIR / f'silhouette_{ds_name}.png', dpi=150)
            plt.close()
    except Exception:
        pass

    if best_k is None:
        best_k = 2

    # Финальный KMeans с лучшим k
    km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
    labels_km = km_final.fit_predict(X_proc)
    metrics_km = compute_scores(X_proc, labels_km)

    # Если признаков мало (<=4), используем иерархическую кластеризацию
    if X_proc.shape[1] <= 4:
        alt = AgglomerativeClustering(n_clusters=best_k, linkage='average')
        labels_alt = alt.fit_predict(X_proc)
    else:
        # Для многомерных данных — DBSCAN
        alt = DBSCAN(eps=0.5, min_samples=5)
        labels_alt = alt.fit_predict(X_proc)

    metrics_alt = compute_scores(X_proc, labels_alt)

    summaries[ds_name] = {
        'best_k': int(best_k),
        'k_silhouette': scores_by_k,
        'kmeans': metrics_km,
        'alt': metrics_alt,
    }

    # Визуализация PCA
    fname_fig = FIG_DIR / f'pca_{ds_name}.png'
    plot_pca_scatter(X_proc, labels_km, f'PCA {ds_name} KMeans', fname_fig)

    # Сохранение меток (используем исходный df, чтобы сохранить привязку к sample_id)
    fname_labels = LABELS_DIR / f'labels_hw07_{ds_name}.csv'
    save_labels_df(df, labels_km, fname_labels)

    print(f"{ds_name} обработан. График: {fname_fig.name}, Метки: {fname_labels.name}")

metrics_summary = summaries


 --- ds1 ---


,sample_id,f01,f02,f03,f04,f05,f06,f07,f08
0,0,-0.536647,-69.812900,-0.002657,71.743147,-11.396498,-12.291287,-6.836847,-0.504094
1,1,15.230731,52.727216,-1.273634,-104.123302,11.589643,34.316967,-49.468873,0.390356
2,2,18.542693,77.317150,-1.321686,-111.946636,10.254346,25.892951,44.595250,0.325893
3,3,-12.538905,-41.709458,0.146474,16.322124,1.391137,2.014316,-39.930582,0.139297
4,4,-6.903056,61.833444,-0.022466,-42.631335,3.107154,-5.471054,7.001149,0.131213


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   sample_id  12000 non-null  int64  
 1   f01        12000 non-null  float64
 2   f02        12000 non-null  float64
 3   f03        12000 non-null  float64
 4   f04        12000 non-null  float64
 5   f05        12000 non-null  float64
 6   f06        12000 non-null  float64
 7   f07        12000 non-null  float64
 8   f08        12000 non-null  float64
dtypes: float64(8), int64(1)
memory usage: 843.9 KB
None
ds1 обработан. График: pca_ds1.png, Метки: labels_hw07_ds1.csv

 --- ds2 ---


,sample_id,x1,x2,z_noise
0,0,0.098849,-1.846034,21.288122
1,1,-1.024516,1.829616,6.072952
2,2,-1.094178,-0.158545,-18.938342
3,3,-1.612808,-1.565844,-11.629462
4,4,1.659901,-2.133292,1.895472


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   sample_id  8000 non-null   int64  
 1   x1         8000 non-null   float64
 2   x2         8000 non-null   float64
 3   z_noise    8000 non-null   float64
dtypes: float64(3), int64(1)
memory usage: 250.1 KB
None
ds2 обработан. График: pca_ds2.png, Метки: labels_hw07_ds2.csv

 --- ds3 ---


,sample_id,x1,x2,f_corr,f_noise
0,0,-2.710470,4.997107,-1.015703,0.718508
1,1,8.730238,-8.787416,3.953063,-1.105349
2,2,-1.079600,-2.558708,0.976628,-3.605776
3,3,6.854042,1.560181,1.760614,-1.230946
4,4,9.963812,-8.869921,2.966583,0.915899


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   sample_id  15000 non-null  int64  
 1   x1         15000 non-null  float64
 2   x2         15000 non-null  float64
 3   f_corr     15000 non-null  float64
 4   f_noise    15000 non-null  float64
dtypes: float64(4), int64(1)
memory usage: 586.1 KB
None
ds3 обработан. График: pca_ds3.png, Метки: labels_hw07_ds3.csv


In [51]:
# Проверка устойчивости KMeans для ds1
stability_ds1 = None
if not df_1.empty:

    X1 = df_1.copy()
    if 'sample_id' in X1.columns:
        X1 = X1.drop(columns=['sample_id'])

    preproc1, _, _ = build_preprocessor(X1)
    X1_proc = preproc1.fit_transform(X1)

    # Берём best_k из метрик (если есть)
    if 'ds1' in metrics_summary and metrics_summary['ds1'].get('best_k') is not None:
        k_use = int(metrics_summary['ds1']['best_k'])
    else:
        k_use = 2

    labels_list = []
    for seed in range(5):
        # Обучаем KMeans с разными random_state для проверки стабильности
        km = KMeans(n_clusters=k_use, random_state=seed, n_init=10)
        labels_list.append(km.fit_predict(X1_proc))

    # Вычисляем Adjusted Rand Index (ARI) между всеми парами результатов
    aris = []
    for i in range(len(labels_list)):
        for j in range(i+1, len(labels_list)):
            aris.append(float(adjusted_rand_score(labels_list[i], labels_list[j])))
    
    stability_ds1 = {'aris': aris, 'mean_ari': float(np.mean(aris))}
    print(f'Stability ds1 (k={k_use}) mean ARI =', stability_ds1['mean_ari'])
else:
    print('df_1 is empty — skipping stability check')

Stability ds1 (k=2) mean ARI = 1.0


In [52]:
# Сохранение итогов
out_metrics = {'datasets': metrics_summary, 'stability_ds1': stability_ds1}
(ARTIFACTS / 'metrics_summary.json').write_text(json.dumps(out_metrics, indent=2))
best_configs = {}
for k, v in metrics_summary.items():

    if v.get('kmeans') and v['kmeans']['silhouette'] is not None:
        best = 'kmeans'
    else:
        best = 'alt'
    best_configs[k] = {'best': best, 'best_k': v.get('best_k')}

(ARTIFACTS / 'best_configs.json').write_text(json.dumps(best_configs, indent=2))
print('Saved metrics_summary.json and best_configs.json in', ARTIFACTS)

Saved metrics_summary.json and best_configs.json in c:\Users\Maken\Desktop\AI-course\homeworks\HW07\artifacts
